# 🎨 TRELLIS (Image-to-3D) — Google Colab Setup

**Official repo:** https://github.com/microsoft/TRELLIS

This notebook installs TRELLIS the way the **official `setup.sh` script is designed to be used**:
it creates its own dedicated `trellis` conda environment pinned to **Python 3.10**, and installs
the exact pinned `PyTorch 2.4.0 + CUDA 11.8` inside that environment.

**Why this matters on Colab specifically:** Colab's base Python is now **3.13**. PyTorch 2.4.0 and
several of TRELLIS's dependencies (`open3d`, `kaolin`, etc.) don't have wheels for 3.13. Installing
into Colab's base environment (skipping `--new-env`) leads to a cascade of version-mismatch errors.
Using `--new-env` sidesteps all of that, because the official script always targets Python 3.10
regardless of what Colab's base interpreter is.

### How to run this notebook
1. `Runtime → Change runtime type → GPU` (T4/L4/A100 — at least 16GB VRAM recommended).
2. Run cells **top to bottom, in order**.
3. **One cell restarts the kernel automatically** (the `condacolab` install). This is expected —
   just continue running from the next cell down afterward. Do not re-run the condacolab cell.
4. The full setup (CUDA toolkit + conda env + all TRELLIS extensions) takes roughly **20–40 minutes**.


## 1. Check GPU
Make sure a GPU is attached before doing anything else.

In [ ]:
!nvidia-smi

## 2. Install CUDA 11.8 Toolkit (nvcc)

Colab's GPU runtime ships NVIDIA *drivers* but not necessarily a matching CUDA *Toolkit* with `nvcc`.
Several TRELLIS extensions (`diffoctreerast`, `kaolin`, `nvdiffrast`) compile CUDA code at install
time and need `nvcc` from a toolkit that matches the CUDA 11.8 PyTorch build TRELLIS pins to.

This installs CUDA 11.8 **alongside** whatever Colab already has — it does not remove the driver.

In [ ]:
%%bash
set -e

echo '========================================'
echo '🔧 Installing CUDA 11.8 Toolkit (nvcc)'
echo '========================================'

if [ -x /usr/local/cuda-11.8/bin/nvcc ]; then
    echo "✅ CUDA 11.8 toolkit already present, skipping."
else
    apt-get -qq update
    apt-get -qq install -y wget gnupg
    wget -q https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb
    dpkg -i cuda-keyring_1.1-1_all.deb
    apt-get -qq update
    apt-get -qq install -y cuda-toolkit-11-8
fi

export PATH="/usr/local/cuda-11.8/bin:$PATH"
nvcc --version
echo '✅ CUDA 11.8 toolkit ready!'

## 3. Clone the TRELLIS repository
`--recurse-submodules` is required — TRELLIS depends on a couple of git submodules.

In [ ]:
%cd /content
!git clone --recurse-submodules https://github.com/microsoft/TRELLIS.git
%cd /content/TRELLIS

## 4. Install `condacolab`

⚠️ **This cell restarts the Colab kernel automatically as soon as it finishes — that's normal.**
When it restarts, don't re-run this cell. Just continue running the notebook from **Step 5** below.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

---
## ⬇️ Kernel restarted? Continue from here. ⬇️
---
## 5. Verify conda is available and accept the Anaconda channel Terms of Service

Recent conda versions require explicitly accepting ToS for the default `main`/`r` channels before
installing anything from them non-interactively — this only needs to happen once per session.

In [ ]:
import subprocess

def is_conda_installed():
    try:
        result = subprocess.run(['conda', '--version'], capture_output=True, text=True)
        return result.returncode == 0
    except FileNotFoundError:
        return False

assert is_conda_installed(), "❌ Conda not found — did the kernel restart after condacolab.install()? Try Runtime > Restart session, then re-run from this cell."
print("✅ Conda is ready:", subprocess.check_output(['conda', '--version']).decode().strip())

In [ ]:
%%bash
set -e
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main || true
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r || true
echo "✅ Channel ToS accepted"


## 6. Run the official `setup.sh` with `--new-env`

This is the key fix versus a manual/base-env install:

- `--new-env` makes the script create its **own** `trellis` conda environment with **Python 3.10**
  and install the exact pinned `pytorch==2.4.0 torchvision==0.19.0 pytorch-cuda=11.8` combo TRELLIS
  is built and tested against — completely independent of Colab's Python 3.13 base env.
- Because that env is Python 3.10, packages like `open3d` (which has no 3.13 wheels) install cleanly.
- Flags used below match the official README's full-featured install, plus `--demo` for the Gradio app.

This step compiles several CUDA extensions from source and will take **~20–40 minutes**. This is
expected — TRELLIS's README calls this out explicitly ("installation may take a while").

In [ ]:
%%bash
set -e

export PATH="/usr/local/cuda-11.8/bin:$PATH"
export LD_LIBRARY_PATH="/usr/local/cuda-11.8/lib64:${LD_LIBRARY_PATH:-}"
export CUDA_HOME="/usr/local/cuda-11.8"
export TORCH_CUDA_ARCH_LIST="7.0;7.5;8.0;8.6+PTX"

cd /content/TRELLIS

echo '========================================'
echo '🏗️  Creating the "trellis" conda env (Python 3.10) and running setup.sh'
echo '    FLAGS: --new-env --basic --xformers --flash-attn'
echo '           --diffoctreerast --spconv --mipgaussian'
echo '           --kaolin --nvdiffrast --demo'
echo '========================================'

. ./setup.sh \
    --new-env \
    --basic \
    --xformers \
    --flash-attn \
    --diffoctreerast \
    --spconv \
    --mipgaussian \
    --kaolin \
    --nvdiffrast \
    --demo

echo ''
echo '✅ setup.sh complete! The "trellis" conda env is fully set up.'

## 7. Verify the install

Every command from here on must run **inside the `trellis` env**, not Colab's base env — use
`conda run -n trellis ...` (as below) for one-off commands.

In [ ]:
!conda run -n trellis python -c "
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA version:', torch.version.cuda)
    print('GPU:', torch.cuda.get_device_name(0))
import open3d
print('open3d:', open3d.__version__)
"


## 8. Run a minimal image-to-3D example

Uses the repo's own `assets/example_image/T.png`. Swap in your own image path as needed.

In [ ]:
%%bash
set -e
cd /content/TRELLIS
export PATH="/usr/local/cuda-11.8/bin:$PATH"

conda run -n trellis python example.py

ls -la sample_gs.mp4 sample_rf.mp4 sample_mesh.mp4 sample.glb sample.ply 2>/dev/null || true
echo '✅ Example run complete — check /content/TRELLIS for the output files.'

## 9. (Optional) Launch the Gradio web demo

Runs `app.py` with a public `share=True` link so you can use it from the browser. Stop the cell to
shut the demo down.

If your GPU doesn't support `flash-attn` (e.g. an older T4/V100), uncomment the `ATTN_BACKEND` line
to fall back to `xformers` — see the official README's note on this.

In [ ]:
%%bash
cd /content/TRELLIS
export PATH="/usr/local/cuda-11.8/bin:$PATH"

# export ATTN_BACKEND=xformers   # uncomment if your GPU doesn't support flash-attn

conda run -n trellis python app.py --server_port 7860 --share